 # GLayout Cascode OpAmp Demo

<a href="https://colab.research.google.com/github/idea-fasoc/OpenFASOC/blob/main/docs/source/notebooks/glayout/GLayout_Cmirror.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


```
OpenFASOC Team, April 2025
Updated June 2025

SPDX-License-Identifier: Apache-2.0
```

**Note:** This repo is still running on the old code base at https://github.com/idea-fasoc/OpenFASOC. We're currently migrating it to the refactored code base.

## Introduction
Welcome!

This Notebook will run as-is on Google Collab, or you can run locally by using the install steps in [this document](https://docs.google.com/document/d/e/2PACX-1vRL8ksIvB-fHaqWgkgBPDUznOcDmmFhNrvzPNx9GSSkZyfhJYexEI9gBZCJ0SNNnHdUrAf1EBOeU182/pub). If you choose a local install, skip part 1 of this notebook.


## Installation On Google Collab
### 1. Clone the repository and install dependencies
**Python Dependencies**
* [`gdsfactory`](https://github.com/gdsfactory/gdsfactory): Provides the backend for GDS manipulation.
* [`sky130`](https://github.com/gdsfactory/skywater130): The Skywater 130nm PDK Python package for GDSFactory to use in this demo.
* [`gf180`](https://github.com/gdsfactory/gf180): The GF 180nm PDK Python package for GDSFactory to use in this demo.
* [`gdstk`](https://heitzmann.github.io/gdstk/): (installed as a part of gdsfactory) Used for converting GDS files into SVG images for viewing.
* [`svgutils`](https://svgutils.readthedocs.io/en/latest/): To scale the SVG image.

**System Dependencies**
* [`klayout`](https://klayout.de/): For DRC (Design Rule Checking).


#### 1.1. Installing the binary dependency `klayout` using micromamba
**You only need to run this once**

In [ ]:
# Clone OpenFASoC
!cd /content
!git clone https://github.com/idea-fasoc/OpenFASOC
!cd OpenFASOC
!git checkout 334ab7e8b1fce72a6983af6a539adc081ee22294
!cd ..
!wget https://raw.githubusercontent.com/idea-fasoc/OpenFASOC/refs/heads/main/openfasoc/generators/glayout/glayout/flow/blocks/elementary/diff_pair/diff_pair.py

In [ ]:
# Setup the environment for the OpenFASOC GDSFactory generator
# You only need to run this block once!

# Install python dependencies
!pip install sky130
!pip install gf180 prettyprinttree svgutils
!pip install gdsfactory==7.7.0
!pip install google-colab==1.0.0 ipython==7.34.0

import pathlib
import os
# Install KLayout (via conda)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
conda_prefix_path = pathlib.Path('conda-env')
CONDA_PREFIX = str(conda_prefix_path.resolve())
%env CONDA_PREFIX={CONDA_PREFIX}

!bin/micromamba create --yes --prefix $CONDA_PREFIX
# Install from the litex-hub channel
!bin/micromamba install --yes --prefix $CONDA_PREFIX \
                        --channel litex-hub \
                        --channel main \
                        klayout
!pip install "numpy<2"
!pip install svgwrite


#### 1.2. Adding the `klayout` binary to the system path, then goto the GLayout directory
**You need to run this each time you restart the kernel**

In [ ]:
# Setup the environment for the OpenFASOC GDSFactory generator

# Adding micro-mamba binary directory to the PATH
# This directory contains Klayout
import pathlib
import os
import numpy as np
np.float_ = np.float64
conda_prefix_path = pathlib.Path('conda-env')
CONDA_PREFIX = str(conda_prefix_path.resolve())
%env CONDA_PREFIX={CONDA_PREFIX}
# Add conda packages to the PATH
PATH = os.environ['PATH']
%env PATH={PATH}:{CONDA_PREFIX}/bin

%cd /content/OpenFASOC/openfasoc/generators/glayout

#### 1.3. Importing Libraries and Utility Functions

In [ ]:
from glayout.flow.pdk.sky130_mapped import sky130_mapped_pdk as sky130
from glayout.flow.pdk.gf180_mapped  import gf180_mapped_pdk  as gf180
import gdstk
import svgutils.transform as sg
import IPython.display
from IPython.display import clear_output
import ipywidgets as widgets

# Redirect all outputs here
hide = widgets.Output()

def display_gds(gds_file, scale = 3):
  # Generate an SVG image
  top_level_cell = gdstk.read_gds(gds_file).top_level()[0]
  top_level_cell.write_svg('out.svg')
  # Scale the image for displaying
  fig = sg.fromfile('out.svg')
  fig.set_size((str(float(fig.width) * scale), str(float(fig.height) * scale)))
  fig.save('out.svg')

  # Display the image
  IPython.display.display(IPython.display.SVG('out.svg'))

def display_component(component, scale = 3):
  # Save to a GDS file
  with hide:
    component.write_gds("out.gds")
  display_gds('out.gds', scale)


In [ ]:
from xml.etree import ElementTree as ET

def annotate_svg(filename, x, y, width, height, text, output_file):
    # Parse the existing SVG
    tree = ET.parse(filename)
    root = tree.getroot()

    # Get the SVG namespace
    svg_ns = "http://www.w3.org/2000/svg"
    ET.register_namespace('', svg_ns)

    # Create new elements in the SVG namespace
    def svg_tag(tag):
        return f"{{{svg_ns}}}{tag}"

    # Add rectangle
    rect = ET.Element(svg_tag('rect'), {
        'x': str(x),'y': str(y),'width': str(width),'height': str(height),
        'stroke': 'black','fill': 'none','stroke-width': '3'
    })
    root.append(rect)
    rect = ET.Element(svg_tag('rect'), {
        'x': str(x),'y': str(y),'width': str(width),'height': str(height),
        'stroke': 'white','fill': 'none','stroke-width': '1'
    })
    root.append(rect)

  # Estimate text box size (roughly)
    font_size = 8
    padding = 2
    char_width = font_size*0.6  # Approximate width of each character
    text_width = len(text) * char_width + 2 * padding
    text_height = font_size + 2 * padding

    # Text box background
    text_bg = ET.Element(svg_tag('rect'), {
        # 'x': str(x + width + 5),
        # 'y': str(y + height / 2 - text_height / 2),
        'x': str(x + 5),
        'y': str(y + height - text_height / 2 + 5),
        'width': str(text_width),
        'height': str(text_height),
        'fill': 'black',
        # 'rx': '2',  # optional rounded corners
        # 'ry': '2'
    })
    root.append(text_bg)

    # Foreground text
    label = ET.Element(svg_tag('text'), {
        # 'x': str(x + width + 5 + padding),
        # 'y': str(y + height / 2 + font_size / 2 - 1),
        'x': str(x + 5 + padding),
        'y': str(y + height + font_size / 2 - 1 + 5),
        'fill': 'white',
        'font-size': f'{font_size}px',
        'font-family': 'monospace'
    })
    label.text = text
    root.append(label)

    # Write back to file
    tree.write(output_file)


# OpAmp Layout Design using GLayout API

## Overview

This document provides a detailed tutorial on using the GLayout API to design and generate a layout for an OpAmp circuit. It outlines the steps to initialize the layout components, place the devices, make necessary adjustments, route the connections, and generate the final layout for visualization.

## Prerequisites

Ensure that you have the following installed:
- Python environment with access to GLayout API.
- GDSFactory package for handling generic layout components.
- A PDK (Process Design Kit) appropriate for the technology node you're working on (e.g., `gf180` in this example).

## Step-by-Step Guide

### Step 1: Setting up the Environment

**1.1 Import Required Modules**
Begin by importing the necessary classes and functions from the GLayout and GDSFactory packages:



In [ ]:
from glayout.flow.primitives.guardring import tapring
from glayout.flow.primitives.fet import pmos, nmos
from glayout.flow.primitives.mimcap import mimcap
from glayout.flow.pdk.util.comp_utils import evaluate_bbox, prec_center,  center_to_edge_distance, prec_ref_center
from glayout.flow.pdk.mappedpdk import MappedPDK
from glayout.flow.routing.straight_route import straight_route
from glayout.flow.routing.c_route import c_route
from glayout.flow.routing.smart_route import smart_route
from glayout.flow.routing.L_route import L_route
from gdsfactory import Component
from diff_pair import diff_pair

from glayout.flow.placement.two_transistor_interdigitized import two_pfet_interdigitized, two_nfet_interdigitized
from glayout.flow.pdk.gf180_mapped import gf180_mapped_pdk

### Step 2: Defining Placement Helper Functions

**2.1 Initialize the Current Mirror Component**

Glayout is built on top of GDSFactory, a Python library for GDS file generation. GDSFactory provides a number of rich library functions, allowing for high programmability. On top of that Glayout also provides a number of helper functions to assist in placement. These functions utilize GDSFactory and Glayout functions to help us in component placing.

In [ ]:
def moveRightOf(comp1, comp2, spacing):
  comp2.x = -prec_center(comp1)[0]
  comp2.y = -prec_center(comp1)[1]
  comp2.movex(center_to_edge_distance(comp1, "E") + center_to_edge_distance(comp2, "W") + spacing)

def moveBelowOf(comp1, comp2, spacing):
  comp2.x = -prec_center(comp1)[0]
  comp2.y = -prec_center(comp1)[1]
  comp2.movey(-(center_to_edge_distance(comp1, "S") + center_to_edge_distance(comp2, "N") + spacing))

def moveAboveOf(comp1, comp2, spacing):
  comp2.x = -prec_center(comp1)[0]
  comp2.y = -prec_center(comp1)[1]
  comp2.movey((center_to_edge_distance(comp1, "N") + center_to_edge_distance(comp2, "S") + spacing))

def moveLeftOf(comp1, comp2, spacing):
  comp2.x = -prec_center(comp1)[0]
  comp2.y = -prec_center(comp1)[1]
  comp2.movex(-(center_to_edge_distance(comp1, "W") + center_to_edge_distance(comp2, "E") + spacing))

def alignRightEdge(comp1, comp2, spacing):
  if evaluate_bbox(comp1)[0] < evaluate_bbox(comp2)[0]:
      comp2.movex(center_to_edge_distance(comp2, "W") - center_to_edge_distance(comp1, "W") + spacing)
  else:
    comp2.movex(-(center_to_edge_distance(comp1, "W") - center_to_edge_distance(comp2, "W") + spacing))

def alignBotEdge(comp1, comp2, spacing):
  if evaluate_bbox(comp1)[1] < evaluate_bbox(comp2)[1]:
      comp2.movey(center_to_edge_distance(comp2, "S") - center_to_edge_distance(comp1, "S") + spacing)
  else:
    comp2.movey(-(center_to_edge_distance(comp1, "S") - center_to_edge_distance(comp2, "S") + spacing))

def alignTopEdge(comp1, comp2, spacing):
  if evaluate_bbox(comp1)[1] < evaluate_bbox(comp2)[1]:
      comp2.movey(-(center_to_edge_distance(comp2, "S") - center_to_edge_distance(comp1, "S") + spacing))
  else:
    comp2.movey((center_to_edge_distance(comp1, "S") - center_to_edge_distance(comp2, "S") + spacing))

### Step 3: Defining Cells

**3.1 Create components**

In [ ]:
def pmosCurrentMirror4t(pdk: MappedPDK, width=2):
  currMirrComp = Component()
  PM14 = pmos(pdk, with_substrate_tap=False, width=width, with_dummy=(True, False))
  PM15 = pmos(pdk, with_substrate_tap=False, width=width, with_dummy=(False, False))
  PM16 = pmos(pdk, with_substrate_tap=False, width=width, with_dummy=(False, False))
  PM17 = pmos(pdk, with_substrate_tap=False, width=width, with_dummy=(False, True))

  PM14_ref = currMirrComp << PM14
  PM15_ref = currMirrComp << PM15
  PM16_ref = currMirrComp << PM16
  PM17_ref = currMirrComp << PM17

  acc = center_to_edge_distance(PM14, "E") + center_to_edge_distance(PM15, "W") + pdk.util_max_metal_seperation()
  PM15_ref.movex(acc)
  acc += center_to_edge_distance(PM15, "E") + center_to_edge_distance(PM16, "W") + pdk.util_max_metal_seperation()
  PM16_ref.movex(acc)
  acc += center_to_edge_distance(PM16, "E") + center_to_edge_distance(PM17, "W") + pdk.util_max_metal_seperation()
  PM17_ref.movex(acc)

  tap_ring = tapring(pdk, enclosed_rectangle=evaluate_bbox(currMirrComp.flatten(), padding=pdk.get_grule("nwell", "active_diff")["min_enclosure"]))
  shift_amount = -prec_center(currMirrComp.flatten())[0]
  tring_ref = currMirrComp << tap_ring
  tring_ref.movex(destination=shift_amount)

  currMirrComp << straight_route(pdk, PM14_ref.ports["multiplier_0_gate_E"], PM15_ref.ports["multiplier_0_gate_E"])
  currMirrComp << straight_route(pdk, PM14_ref.ports["multiplier_0_gate_E"], PM16_ref.ports["multiplier_0_gate_E"])
  currMirrComp << straight_route(pdk, PM14_ref.ports["multiplier_0_gate_E"], PM17_ref.ports["multiplier_0_gate_E"])
  currMirrComp << c_route(pdk, PM14_ref.ports["multiplier_0_gate_W"], PM14_ref.ports["multiplier_0_drain_W"])

  currMirrComp.add_ports(PM14_ref.get_ports_list(),prefix=f"PM14_")
  currMirrComp.add_ports(PM15_ref.get_ports_list(),prefix=f"PM15_")
  currMirrComp.add_ports(PM16_ref.get_ports_list(),prefix=f"PM16_")
  currMirrComp.add_ports(PM17_ref.get_ports_list(),prefix=f"PM17_")
  return currMirrComp.flatten()

In [ ]:
display_component(pmosCurrentMirror4t(gf180, width=10), scale=1)


In [ ]:
def pmosCurrentMirror2t(pdk: MappedPDK):
  currMirrComp = Component()
  PM13 = pmos(pdk, with_substrate_tap=False, with_dummy=(True, False))
  PM12 = pmos(pdk, with_substrate_tap=False, with_dummy=(False, True))

  PM13_ref = currMirrComp << PM13
  PM12_ref = currMirrComp << PM12

  acc = center_to_edge_distance(PM13, "E") + center_to_edge_distance(PM12, "W") + pdk.util_max_metal_seperation()
  PM12_ref.movex(acc)

  tap_ring = tapring(pdk, enclosed_rectangle=evaluate_bbox(currMirrComp.flatten(), padding=pdk.get_grule("nwell", "active_diff")["min_enclosure"]))
  shift_amount = -prec_center(currMirrComp.flatten())[0]
  tring_ref = currMirrComp << tap_ring
  tring_ref.movex(destination=shift_amount)

  currMirrComp << straight_route(pdk, PM13_ref.ports["multiplier_0_gate_E"], PM12_ref.ports["multiplier_0_gate_E"])
  currMirrComp << c_route(pdk, PM13_ref.ports["multiplier_0_gate_E"], PM13_ref.ports["multiplier_0_drain_E"])

  currMirrComp.add_ports(PM13_ref.get_ports_list(),prefix=f"PM13_")
  currMirrComp.add_ports(PM12_ref.get_ports_list(),prefix=f"PM12_")
  return currMirrComp.flatten()

In [ ]:
display_component(pmosCurrentMirror2t(gf180), scale=1)

In [ ]:
def nmosCurrentMirror4t(pdk: MappedPDK):
  currMirrComp = Component()
  nfet_ref = nmos(pdk, with_dnwell=False, with_substrate_tap=False, with_dummy=(True, False))
  nfet_mir0 = nmos(pdk, with_dnwell=False, with_substrate_tap=False, with_dummy=(False, False))
  nfet_mir1 = nmos(pdk, with_dnwell=False, with_substrate_tap=False, with_dummy=(False, False))
  nfet_mir2 = nmos(pdk, with_dnwell=False, with_substrate_tap=False, with_dummy=(False, True))

  cref_ref = currMirrComp << nfet_ref
  cmir0_ref = currMirrComp << nfet_mir0
  cmir1_ref = currMirrComp << nfet_mir1
  cmir2_ref = currMirrComp << nfet_mir2

  acc = center_to_edge_distance(nfet_ref, "E") + center_to_edge_distance(nfet_mir0, "W") + pdk.util_max_metal_seperation()
  cmir0_ref.movex(acc)
  acc += center_to_edge_distance(nfet_mir0, "E") + center_to_edge_distance(nfet_mir1, "W") + pdk.util_max_metal_seperation()
  cmir1_ref.movex(acc)
  acc += center_to_edge_distance(nfet_mir1, "E") + center_to_edge_distance(nfet_mir2, "W") + pdk.util_max_metal_seperation()
  cmir2_ref.movex(acc)

  tap_ring = tapring(pdk, enclosed_rectangle=evaluate_bbox(currMirrComp.flatten(), padding=pdk.get_grule("nwell", "active_diff")["min_enclosure"]))
  shift_amount = -prec_center(currMirrComp.flatten())[0]
  tring_ref = currMirrComp << tap_ring
  tring_ref.movex(destination=shift_amount)

  currMirrComp << straight_route(pdk, cref_ref.ports["multiplier_0_gate_E"], cmir0_ref.ports["multiplier_0_gate_E"])
  currMirrComp << straight_route(pdk, cref_ref.ports["multiplier_0_gate_E"], cmir1_ref.ports["multiplier_0_gate_E"])
  currMirrComp << straight_route(pdk, cref_ref.ports["multiplier_0_gate_E"], cmir2_ref.ports["multiplier_0_gate_E"])
  currMirrComp << c_route(pdk, cref_ref.ports["multiplier_0_gate_W"], cref_ref.ports["multiplier_0_drain_W"])

  currMirrComp.add_ports(cref_ref.get_ports_list(),prefix=f"NM10_")
  currMirrComp.add_ports(cmir0_ref.get_ports_list(),prefix=f"NM13_")
  currMirrComp.add_ports(cmir1_ref.get_ports_list(),prefix=f"NM11_")
  currMirrComp.add_ports(cmir2_ref.get_ports_list(),prefix=f"NM12_")
  return currMirrComp

In [ ]:
display_component(nmosCurrentMirror4t(gf180), scale=1)

In [ ]:
def nmos4diodeConnected(pdk: MappedPDK):
  nmos4diodes = Component()
  NM9 = nmos(pdk, with_dnwell=False, with_substrate_tap=False, with_dummy=(False, False))
  NM8 = nmos(pdk, with_dnwell=False, with_substrate_tap=False, with_dummy=(False, False))
  NM7 = nmos(pdk, with_dnwell=False, with_substrate_tap=False, with_dummy=(False, False))
  NM6 = nmos(pdk, with_dnwell=False, with_substrate_tap=False, with_dummy=(False, False))

  NM9_ref = nmos4diodes << NM9
  NM8_ref = nmos4diodes << NM8
  NM7_ref = nmos4diodes << NM7
  NM6_ref = nmos4diodes << NM6

  gap = pdk.util_max_metal_seperation()
  NM8_ref.movey(-(evaluate_bbox(NM9)[1] +   gap))
  NM7_ref.movex(  evaluate_bbox(NM9)[0] + 3*gap )
  NM6_ref.movex(  evaluate_bbox(NM9)[0] + 3*gap )
  NM6_ref.movey(-(evaluate_bbox(NM9)[1] +   gap))

  nmos4diodes << c_route(pdk, NM9_ref.ports["multiplier_0_gate_W"], NM9_ref.ports["multiplier_0_drain_W"])
  nmos4diodes << c_route(pdk, NM8_ref.ports["multiplier_0_gate_W"], NM8_ref.ports["multiplier_0_drain_W"])
  nmos4diodes << c_route(pdk, NM7_ref.ports["multiplier_0_gate_W"], NM7_ref.ports["multiplier_0_drain_W"])
  nmos4diodes << c_route(pdk, NM6_ref.ports["multiplier_0_gate_W"], NM6_ref.ports["multiplier_0_drain_W"])

  nmos4diodes << c_route(pdk, NM9_ref.ports["multiplier_0_source_E"], NM8_ref.ports["multiplier_0_drain_E"], extension=pdk.util_max_metal_seperation())
  nmos4diodes << c_route(pdk, NM7_ref.ports["multiplier_0_source_E"], NM6_ref.ports["multiplier_0_drain_E"], extension=pdk.util_max_metal_seperation())

  nmos4diodes.add_ports(NM9_ref.get_ports_list(),prefix=f"NM9_")
  nmos4diodes.add_ports(NM8_ref.get_ports_list(),prefix=f"NM8_")
  nmos4diodes.add_ports(NM7_ref.get_ports_list(),prefix=f"NM7_")
  nmos4diodes.add_ports(NM6_ref.get_ports_list(),prefix=f"NM6_")
  return nmos4diodes

In [ ]:
display_component(nmos4diodeConnected(gf180), scale=1)

In [ ]:
def pmos4diodeConnected(pdk: MappedPDK):
  pmos4diodes = Component()
  PM11 = pmos(pdk, with_substrate_tap=False, with_dummy=(False, False))
  PM10 = pmos(pdk, with_substrate_tap=False, with_dummy=(False, False))
  PM8 = pmos(pdk, with_substrate_tap=False, with_dummy=(False, False))
  PM9 = pmos(pdk, with_substrate_tap=False, with_dummy=(False, False))

  PM11_ref = pmos4diodes << PM11
  PM10_ref = pmos4diodes << PM10
  PM8_ref = pmos4diodes << PM8
  PM9_ref = pmos4diodes << PM9

  PM10_ref.movey(-(evaluate_bbox(PM11)[1] + pdk.util_max_metal_seperation()))
  PM8_ref.movex(evaluate_bbox(PM11)[0] + 3*pdk.util_max_metal_seperation())
  PM9_ref.movex(evaluate_bbox(PM11)[0] + 3*pdk.util_max_metal_seperation())
  PM9_ref.movey(-(evaluate_bbox(PM11)[1] + pdk.util_max_metal_seperation()))

  pmos4diodes << c_route(pdk, PM11_ref.ports["multiplier_0_gate_E"], PM11_ref.ports["multiplier_0_drain_E"], extension=1.5)
  pmos4diodes << c_route(pdk, PM10_ref.ports["multiplier_0_gate_W"], PM10_ref.ports["multiplier_0_drain_W"])
  pmos4diodes << c_route(pdk, PM8_ref.ports["multiplier_0_gate_E"], PM8_ref.ports["multiplier_0_drain_E"], extension=1.5)
  pmos4diodes << c_route(pdk, PM9_ref.ports["multiplier_0_gate_W"], PM9_ref.ports["multiplier_0_drain_W"])

  pmos4diodes << c_route(pdk, PM10_ref.ports["multiplier_0_source_E"], PM11_ref.ports["multiplier_0_drain_E"], extension=1.5)
  pmos4diodes << c_route(pdk, PM9_ref.ports["multiplier_0_source_E"], PM8_ref.ports["multiplier_0_drain_E"], extension=1.5)

  pmos4diodes.add_ports(PM11_ref.get_ports_list(),prefix=f"PM11_")
  pmos4diodes.add_ports(PM10_ref.get_ports_list(),prefix=f"PM10_")
  pmos4diodes.add_ports(PM8_ref.get_ports_list(),prefix=f"PM8_")
  pmos4diodes.add_ports(PM9_ref.get_ports_list(),prefix=f"PM9_")
  return pmos4diodes.flatten()

In [ ]:
display_component(pmos4diodeConnected(gf180), scale=1)

In [ ]:
def pmosStack(pdk: MappedPDK):
  pmosStack = Component()
  topStack = two_pfet_interdigitized(pdk, 2, with_substrate_tap=False)
  midStack = two_pfet_interdigitized(pdk, 2, with_substrate_tap=False)
  botStack = two_pfet_interdigitized(pdk, 2, with_substrate_tap=False)
  C1 = mimcap(pdk)
  C0 = mimcap(pdk)
  pmosOut = pmos(pdk, with_substrate_tap=False, with_dummy=(False, False))

  topStack_ref = pmosStack << topStack
  botStack_ref = pmosStack << botStack
  midStack_ref = pmosStack << midStack
  C1_ref = pmosStack << C1
  C0_ref = pmosStack << C0
  pmosOut_ref = pmosStack << pmosOut

  moveBelowOf(topStack_ref, midStack_ref, pdk.util_max_metal_seperation())
  moveBelowOf(midStack_ref, botStack_ref, pdk.util_max_metal_seperation())
  moveAboveOf(topStack_ref, pmosOut_ref, pdk.util_max_metal_seperation())
  moveAboveOf(pmosOut_ref, C1_ref, pdk.util_max_metal_seperation())
  moveAboveOf(C1_ref, C0_ref,  pdk.util_max_metal_seperation())

  pmosStack << smart_route(pdk, topStack_ref.ports["B_gate_E"], topStack_ref.ports["A_gate_E"])
  pmosStack << smart_route(pdk, botStack_ref.ports["B_gate_E"], botStack_ref.ports["A_gate_E"])
  pmosStack << smart_route(pdk, midStack_ref.ports["B_gate_E"], midStack_ref.ports["A_gate_E"])

  pmosStack << c_route(pdk, topStack_ref.ports["A_drain_E"], midStack_ref.ports["A_source_E"], extension=3*pdk.util_max_metal_seperation())
  pmosStack << c_route(pdk, topStack_ref.ports["B_drain_W"], midStack_ref.ports["B_source_W"], extension=pdk.util_max_metal_seperation())
  pmosStack << c_route(pdk, midStack_ref.ports["A_drain_E"], botStack_ref.ports["A_source_E"], extension=6*pdk.util_max_metal_seperation())
  pmosStack << c_route(pdk, midStack_ref.ports["B_drain_W"], botStack_ref.ports["B_source_W"], extension=4*pdk.util_max_metal_seperation())

  pmosStack << c_route(pdk, topStack_ref.ports["B_gate_W"], midStack_ref.ports["A_drain_W"], extension=8*pdk.util_max_metal_seperation())

  pmosStack << c_route(pdk, topStack_ref.ports["B_drain_W"], C0_ref.ports["top_met_W"], extension=6*pdk.util_max_metal_seperation())
  pmosStack << c_route(pdk, botStack_ref.ports["B_source_E"], pmosOut_ref.ports["multiplier_0_gate_E"], extension=12*pdk.util_max_metal_seperation())
  pmosStack << straight_route(pdk, C0_ref.ports["bottom_met_S"], C1_ref.ports["bottom_met_S"])

  pmosStack << c_route(pdk, pmosOut_ref.ports["multiplier_0_gate_E"], C1_ref.ports["top_met_E"])
  pmosStack << c_route(pdk, pmosOut_ref.ports["multiplier_0_drain_W"], C1_ref.ports["bottom_met_W"])

  pmosStack.add_ports(topStack_ref.get_ports_list(),prefix=f"topStack_")
  pmosStack.add_ports(botStack_ref.get_ports_list(),prefix=f"botStack_")
  pmosStack.add_ports(midStack_ref.get_ports_list(),prefix=f"midStack_")
  pmosStack.add_ports(C1_ref.get_ports_list(),prefix=f"C1_")
  pmosStack.add_ports(C0_ref.get_ports_list(),prefix=f"C0_")
  pmosStack.add_ports(pmosOut_ref.get_ports_list(),prefix=f"pmosOut_")
  return pmosStack

In [ ]:
display_component(pmosStack(gf180), scale=0.5)

In [ ]:
def nmosStack(pdk: MappedPDK):
  nmosStack = Component()
  topStack = two_nfet_interdigitized(pdk, 2, with_substrate_tap=False)
  midStack = two_nfet_interdigitized(pdk, 2, with_substrate_tap=False)
  botStack = two_nfet_interdigitized(pdk, 2, with_substrate_tap=False)
  C1 = mimcap(pdk)
  C0 = mimcap(pdk)
  pmosOut = nmos(pdk, with_dnwell=False, with_substrate_tap=False, with_dummy=(False, False))

  topStack_ref = nmosStack << topStack
  botStack_ref = nmosStack << botStack
  midStack_ref = nmosStack << midStack
  C1_ref = nmosStack << C1
  C0_ref = nmosStack << C0
  pmosOut_ref = nmosStack << pmosOut

  moveBelowOf(topStack_ref, midStack_ref, pdk.util_max_metal_seperation())
  moveBelowOf(midStack_ref, botStack_ref, pdk.util_max_metal_seperation())
  moveAboveOf(topStack_ref, pmosOut_ref, pdk.util_max_metal_seperation())
  moveAboveOf(pmosOut_ref, C1_ref, pdk.util_max_metal_seperation())
  moveAboveOf(C1_ref, C0_ref,  pdk.util_max_metal_seperation())

  nmosStack << smart_route(pdk, topStack_ref.ports["B_gate_E"], topStack_ref.ports["A_gate_E"])
  nmosStack << smart_route(pdk, botStack_ref.ports["B_gate_E"], botStack_ref.ports["A_gate_E"])
  nmosStack << smart_route(pdk, midStack_ref.ports["B_gate_E"], midStack_ref.ports["A_gate_E"])

  nmosStack << c_route(pdk, topStack_ref.ports["A_drain_E"], midStack_ref.ports["A_source_E"], extension=3*pdk.util_max_metal_seperation())
  nmosStack << c_route(pdk, topStack_ref.ports["B_drain_W"], midStack_ref.ports["B_source_W"], extension=pdk.util_max_metal_seperation())
  nmosStack << c_route(pdk, midStack_ref.ports["A_drain_E"], botStack_ref.ports["A_source_E"], extension=6*pdk.util_max_metal_seperation())
  nmosStack << c_route(pdk, midStack_ref.ports["B_drain_W"], botStack_ref.ports["B_source_W"], extension=4*pdk.util_max_metal_seperation())

  nmosStack << c_route(pdk, topStack_ref.ports["B_gate_W"], midStack_ref.ports["A_drain_W"], extension=8*pdk.util_max_metal_seperation())

  nmosStack << c_route(pdk, topStack_ref.ports["B_drain_W"], C0_ref.ports["top_met_W"], extension=6*pdk.util_max_metal_seperation())
  nmosStack << c_route(pdk, botStack_ref.ports["B_source_E"], pmosOut_ref.ports["multiplier_0_gate_E"], extension=12*pdk.util_max_metal_seperation())
  nmosStack << straight_route(pdk, C0_ref.ports["bottom_met_S"], C1_ref.ports["bottom_met_S"])

  nmosStack << c_route(pdk, pmosOut_ref.ports["multiplier_0_gate_E"], C1_ref.ports["top_met_E"])
  nmosStack << c_route(pdk, pmosOut_ref.ports["multiplier_0_drain_W"], C1_ref.ports["bottom_met_W"])

  nmosStack.add_ports(topStack_ref.get_ports_list(),prefix=f"topStack_")
  nmosStack.add_ports(botStack_ref.get_ports_list(),prefix=f"botStack_")
  nmosStack.add_ports(midStack_ref.get_ports_list(),prefix=f"midStack_")
  nmosStack.add_ports(C1_ref.get_ports_list(),prefix=f"C1_")
  nmosStack.add_ports(C0_ref.get_ports_list(),prefix=f"C0_")
  nmosStack.add_ports(pmosOut_ref.get_ports_list(),prefix=f"pmosOut_")
  return nmosStack

In [ ]:
display_component(nmosStack(gf180), scale=0.5)

In [ ]:
def stack(pdk: MappedPDK):
  stack = Component()

  pmosStack_inst = pmosStack(pdk)
  nmosStack_inst = nmosStack(pdk)

  pmosStack_ref = stack << pmosStack_inst
  nmosStack_ref = stack << nmosStack_inst

  moveBelowOf(nmosStack_ref, pmosStack_ref, 2*pdk.util_max_metal_seperation())

  stack << c_route(pdk, nmosStack_ref.ports["midStack_A_drain_W"], pmosStack_ref.ports["botStack_A_drain_W"], extension=20*pdk.util_max_metal_seperation())
  stack << c_route(pdk, nmosStack_ref.ports["midStack_B_drain_E"], pmosStack_ref.ports["botStack_B_drain_E"], extension=20*pdk.util_max_metal_seperation())

  stack << c_route(pdk, nmosStack_ref.ports["botStack_A_drain_W"], pmosStack_ref.ports["midStack_A_drain_W"], extension=30*pdk.util_max_metal_seperation())
  stack << c_route(pdk, nmosStack_ref.ports["botStack_B_drain_E"], pmosStack_ref.ports["midStack_B_drain_E"], extension=30*pdk.util_max_metal_seperation())

  stack << c_route(pdk, nmosStack_ref.ports["pmosOut_multiplier_0_drain_E"], pmosStack_ref.ports["pmosOut_multiplier_0_drain_E"], extension=50*pdk.util_max_metal_seperation())

  stack.add_ports(pmosStack_ref.get_ports_list(),prefix=f"pmosStack_")
  stack.add_ports(nmosStack_ref.get_ports_list(),prefix=f"nmosStack_")
  return stack

In [ ]:
display_component(stack(gf180), scale=0.5)

In [ ]:
def opamp(pdk: MappedPDK):
  opamp = Component()
  pmosCurrentMirror4t_inst = pmosCurrentMirror4t(pdk)
  nmos4diodeConnected_inst = nmos4diodeConnected(pdk)
  nmosCurrentMirror4t_inst = nmosCurrentMirror4t(pdk)
  pmosCurrentMirror2t_inst = pmosCurrentMirror2t(pdk)
  pmos4diodeConnected_inst = pmos4diodeConnected(pdk)
  diffpair_inst = diff_pair(pdk, n_or_p_fet=False)
  stack_inst = stack(pdk)

  pmosCurrentMirror4t_ref = prec_ref_center(pmosCurrentMirror4t_inst)
  nmos4diodeConnected_ref = prec_ref_center(nmos4diodeConnected_inst)
  nmosCurrentMirror4t_ref = prec_ref_center(nmosCurrentMirror4t_inst)
  pmosCurrentMirror2t_ref = prec_ref_center(pmosCurrentMirror2t_inst)
  pmos4diodeConnected_ref = prec_ref_center(pmos4diodeConnected_inst)
  diffpair_ref = prec_ref_center(diffpair_inst)
  stack_ref = prec_ref_center(stack_inst)

  opamp.add(pmosCurrentMirror4t_ref)
  opamp.add(nmos4diodeConnected_ref)
  opamp.add(nmosCurrentMirror4t_ref)
  opamp.add(pmosCurrentMirror2t_ref)
  opamp.add(pmos4diodeConnected_ref)
  opamp.add(diffpair_ref)
  opamp.add(stack_ref)

  moveBelowOf(pmosCurrentMirror4t_ref, nmos4diodeConnected_ref, pdk.util_max_metal_seperation())
  moveRightOf(pmosCurrentMirror4t_ref, diffpair_ref, pdk.util_max_metal_seperation())
  moveRightOf(diffpair_ref, nmosCurrentMirror4t_ref, pdk.util_max_metal_seperation())
  alignTopEdge(pmosCurrentMirror4t_ref, diffpair_ref, 0)
  moveBelowOf(nmosCurrentMirror4t_ref, pmos4diodeConnected_ref, pdk.util_max_metal_seperation())
  moveAboveOf(diffpair_ref, pmosCurrentMirror2t_ref, pdk.util_max_metal_seperation())
  moveBelowOf(diffpair_ref, stack_ref, pdk.util_max_metal_seperation())

  opamp << c_route(pdk, pmosCurrentMirror4t_ref.ports["PM15_multiplier_0_drain_E"], nmos4diodeConnected_ref.ports["NM9_multiplier_0_drain_E"])
  opamp << c_route(pdk, pmosCurrentMirror4t_ref.ports["PM16_multiplier_0_drain_E"], nmos4diodeConnected_ref.ports["NM7_multiplier_0_drain_E"])

  opamp << c_route(pdk, nmosCurrentMirror4t_ref.ports["NM13_multiplier_0_drain_W"], pmos4diodeConnected_ref.ports["PM10_multiplier_0_drain_W"])
  opamp << c_route(pdk, nmosCurrentMirror4t_ref.ports["NM12_multiplier_0_drain_E"], pmos4diodeConnected_ref.ports["PM9_multiplier_0_drain_E"])

  opamp << c_route(pdk, nmosCurrentMirror4t_ref.ports["NM11_multiplier_0_drain_E"], pmosCurrentMirror2t_ref.ports["PM12_multiplier_0_gate_E"])

  opamp << straight_route(pdk, pmosCurrentMirror4t_ref.ports["PM17_multiplier_0_drain_W"], nmosCurrentMirror4t_ref.ports["NM10_multiplier_0_drain_W"])

  opamp << c_route(pdk, pmosCurrentMirror2t_ref.ports["PM12_multiplier_0_drain_E"], diffpair_ref.ports["tr_multiplier_0_source_E"])

  opamp << c_route(pdk, diffpair_ref.ports["bl_multiplier_0_drain_W"], stack_ref.ports["nmosStack_topStack_A_drain_W"], extension=20*pdk.util_max_metal_seperation())
  opamp << c_route(pdk, diffpair_ref.ports["br_multiplier_0_drain_E"], stack_ref.ports["nmosStack_topStack_B_drain_E"], extension=20*pdk.util_max_metal_seperation())
  return opamp


In [ ]:
display_component(opamp(gf180), scale=0.5)